## 1. Setup

Library imports and display settings for the image analysis pipeline.

| Library | Role in this notebook |
|---|---|
| `imageio` | Reading microscopy image files |
| `numpy` | Array operations on image data |
| `pandas` | Per-cell and per-object measurement tables |
| `skimage.filters.threshold_li` | Li minimum cross-entropy thresholding for segmentation |
| `scipy.ndimage` | Morphological operations on binary masks |
| `seaborn`, `matplotlib` | Figure generation |
| `os`, `shutil`, `subprocess` | File handling and external tool invocation |
| `pickle` | Serialising intermediate results |

Pandas display options are widened so that per-cell measurement tables print in
full during interactive inspection.

In [ ]:
import os
import numpy as np
import pandas as pd
import imageio
import seaborn as sns
import matplotlib.pyplot as plt
import shutil
import subprocess
import pickle
from skimage.filters import threshold_li
from skimage.filters import threshold_otsu
import scipy
from scipy.ndimage.morphology import binary_dilation
from scipy import stats
pd.set_option('display.max_rows', 700)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)


## 2. Image analysis functions

Extracts per-voxel intensities along the mitochondrial network skeleton
computed by MitoGraph, for colocalisation analysis between channels.

**`runMitoGraph`** — invokes the MitoGraph binary on a folder of TIF stacks with
voxel dimensions xy = 0.07 µm, z = 0.200 µm, then moves its output files into a
`MitoGraphFiles` subfolder.

**`convertIntoPixelUnit`** — reads MitoGraph's skeleton coordinate table and
converts µm coordinates to voxel indices by dividing by the voxel size and
rounding.

**`readIntensitiesInMitoNetwork` / `getIntensities`** — for each skeleton
coordinate, samples channels 2 and 3 and records the mean intensity of the
surrounding 3×3×3 voxel neighbourhood. Images are flipped along the y axis to
match MitoGraph's coordinate convention.

**`main`** — iterates over the Channel 1 TIF files in a folder, building a
per-cell table of skeleton coordinates with matched intensities in the other
channels.

**`getMandersColCoeff`** — Manders' M1: the fraction of total signal in one
channel that falls within the thresholded mask of the other.

**`thresholdScan`** — Li minimum cross-entropy threshold, optionally followed by
one round of binary dilation, returning the intensity-weighted masked signal.

**`get_pvalue` / `get_pvalue_fl`** — Welch-style independent *t*-test comparing
each mutant against the `rho+` reference, returning the p-value as string or
float.

In [ ]:
def main(folder):
    all_cells = {}
    for file in os.listdir(folder + 'Channel_1/'):
        
        if file.endswith(".tif"):
            filename = file[:-9]
            print(file , "+", file[:-9])
            df = convertIntoPixelUnit(folder, filename, 2, file[-7:-4])
            df = readIntensitiesInMitoNetwork(df, folder + 'Channel_', file)
            all_cells[file] = df
    return all_cells

def runMitoGraph(folder):
    '''takes a folder name as input and runs the MitoGraph software on this folder '''
    os.chdir('/home/huygens/Documents/Microscopy')    
    command = 'MitoGraph -xy 0.11 -z 0.200 -path ' + folder
    
    output=subprocess.check_output(command, stderr=subprocess.STDOUT, shell=True)
    os.mkdir(os.path.join(folder,'MitoGraphFiles'))
    
    for file in os.listdir(folder):
        if not file.endswith('.tif') and file != 'MitoGraphFiles':
            shutil.move(folder + '/' + file, folder + '/MitoGraphFiles/')
            
    return output  


def readIntensitiesInMitoNetwork(df, folder, filename2, xydim=0.11, zdim=0.2):
    for j in [2,3]:
        file2 = folder + str(j) + '/' + filename2[:-9] + str(j) + filename2[-8:-4] + '.tif'
        df['Intensity ' + str(j)] = getIntensities(file2, j, df)
    return df
    
def getIntensities(file, channel, df):
    im = np.flip(np.array(imageio.mimread(file)),1)
    return [im[df['alt_z'][i]-1:df['alt_z'][i]+2, df['alt_y'][i]-1:df['alt_y'][i]+2,df['alt_x'][i]-1:df['alt_x'][i]+2].mean() for i in range(df.shape[0])]
    
def convertIntoPixelUnit(folder, filename, channel, cellNumber, xydim=0.11, zdim=0.2):
    df = pd.read_table(folder + "Channel_" + str(channel) + '/MitoGraphFiles/' + filename + str(channel) + '_'+ cellNumber + '.txt')
    # the following converts the µm unit into pixel unit
    df['alt_x'] = round(df['x']/xydim).astype('int32')
    df['alt_y'] = round(df['y']/xydim).astype('int32')
    df['alt_z'] = round(df['z']/zdim).astype('int32')
    return df



def getMandersColCoeff(array_one, array_two):
    numerator = sum([array_one[i] for i in range(array_one.size) if array_two[i] > 0])
    denominator = array_one.sum()
    m1 = numerator/denominator
    return m1



def thresholdScan(lineScan, dilate=True):    
    thresh = threshold_li(lineScan)
    binary = lineScan > thresh
    
    if dilate:
        binary = binary_dilation(binary)
    
    return lineScan * binary

def get_pvalue(df, mutant):
    #statistic, pvalue = scipy.stats.mannwhitneyu(df['wt'], df[mutant])
    statistic, pvalue = scipy.stats.ttest_ind(df["rho+"], df[mutant], nan_policy='omit')
    return str(float(round(pvalue, 6)))
def get_pvalue_fl(df, mutant):
    #statistic, pvalue = scipy.stats.mannwhitneyu(df['wt'], df[mutant])
    statistic, pvalue = scipy.stats.ttest_ind(df["rho+"], df[mutant], nan_policy='omit')
    return float(round(pvalue, 6))

## 3. Run MitoGraph

Runs MitoGraph on the Channel 2 image stacks to skeletonise the mitochondrial
network in 3D. Output files are written alongside the images and then moved into
a `MitoGraphFiles` subfolder.

This step calls an external binary and is run once per dataset; downstream cells
read its output tables.

**Requires:** MitoGraph on the system `PATH`.
Voxel dimensions passed: xy = 0.07 µm, z = 0.200 µm.

In [ ]:
# Directory containing the per-channel image folders (Channel_1/, Channel_2/, ...).
IMAGE_DIR = '.'
runMitoGraph(f'{IMAGE_DIR}/Channel_2')

## 4. Extract per-cell skeleton measurements

Runs the extraction over every cell in the dataset. `main` iterates the Channel 1
TIF files, reads the matching MitoGraph skeleton table for Channel 2, converts
its µm coordinates to voxel indices, and samples the mean intensity of Channels
2 and 3 in a 3×3×3 neighbourhood around each skeleton node.

The result is a dictionary keyed by filename, each value a per-node table for one
cell: skeleton coordinates plus matched intensities in both channels.



In [ ]:
# Root folder containing Channel_1/, Channel_2/, Channel_3/ subfolders.
folder = f"{IMAGE_DIR}/"
all_cells = main(folder)

In [ ]:
all_cells = main(folder)

20260708_su9kate_pda1NG_cap_series01_T16_decon_1_026.tif + 20260708_su9kate_pda1NG_cap_series01_T16_decon_
20260708_su9kate_pda1NG_cap_series01_T16_decon_1_027.tif + 20260708_su9kate_pda1NG_cap_series01_T16_decon_
20260708_su9kate_pda1NG_cap_series01_T16_decon_1_028.tif + 20260708_su9kate_pda1NG_cap_series01_T16_decon_
20260708_su9kate_pda1NG_cap_series01_T16_decon_1_029.tif + 20260708_su9kate_pda1NG_cap_series01_T16_decon_
20260708_su9kate_pda1NG_cap_series01_T16_decon_1_030.tif + 20260708_su9kate_pda1NG_cap_series01_T16_decon_
20260708_su9kate_pda1NG_cap_series01_T16_decon_1_031.tif + 20260708_su9kate_pda1NG_cap_series01_T16_decon_
20260708_su9kate_pda1NG_cap_series01_T16_decon_1_032.tif + 20260708_su9kate_pda1NG_cap_series01_T16_decon_
20260708_su9kate_pda1NG_cap_series01_T16_decon_1_033.tif + 20260708_su9kate_pda1NG_cap_series01_T16_decon_
20260708_su9kate_pda1NG_cap_series01_T16_decon_1_035.tif + 20260708_su9kate_pda1NG_cap_series01_T16_decon_
20260708_su9kate_pda1NG_cap_series01_

## 5. Per-cell colocalisation coefficients

Computes two colocalisation measures per cell from the intensities sampled along
the mitochondrial skeleton, producing one row per cell.

- **Manders' M1** — the fraction of Channel 2 signal that falls at skeleton
  positions where Channel 3 is above its threshold. Both channels are
  thresholded with Li minimum cross-entropy first (`dilate=False` here, so no
  mask expansion is applied).
- **Pearson correlation** — linear correlation between the two channels'
  intensities across all skeleton nodes, computed on the unthresholded values.

The two are complementary: Pearson measures whether the intensities co-vary,
Manders measures what fraction of one signal overlaps the other, and they can
disagree when one channel is present at a roughly constant level.

In [ ]:
results = pd.DataFrame()
for key in all_cells.keys():
    temp_df = all_cells[key]
    correlation_df = {}
    correlation_df['Cell'] = key
    array_one = np.array(temp_df['Intensity 2'])
    array_two = np.array(temp_df['Intensity 3'])
    one = thresholdScan(array_one, dilate = False)
    two = thresholdScan(array_two, dilate = False)
    correlation_df['_manders'] = (getMandersColCoeff(one, two))
    correlation_df['_pearson'] = temp_df['Intensity 2'].corr(temp_df['Intensity 3'])
    results = results.append(correlation_df, ignore_index=True)
results.index = results['Cell']
results = results.iloc[:, 1:]
df2 = results

## 6. Timelapse analysis functions

The timelapse dataset requires different treatment from the fixed-cell analysis:
the same cell is imaged repeatedly, so measurements must be linked across frames
and must not be confounded by frame-to-frame changes in the analysis itself.

**`get_cell_id`** — strips the timepoint tag (`_T0_`, `_T1_`, …) from a filename
so that all frames of one tracked cell map to a single identifier.

**`get_fixed_thresholds`** — computes one threshold per cell from that cell's
baseline (T0) frame only, and reuses it for every subsequent timepoint. This is
the key design decision of the timelapse analysis: an adaptive threshold
recomputed per frame would move with the signal it is measuring, so any real
change over time would be partly absorbed by the threshold and the measurement
would understate it. Fixing the threshold at baseline means later frames are
scored against a constant criterion.

**`getMandersColCoeff_fixed`** — Manders' M2: the fraction of mitochondrial
signal located at positions where the second channel exceeds the fixed
threshold. The mitochondrial channel is used raw rather than re-thresholded,
since the sampled coordinates already lie on the MitoGraph skeleton and are
therefore inside the network by construction.

**`gini` and `coefficient_of_variation`** — threshold-free measures of how
unevenly the signal is distributed along the network. Gini approaches 0 for a
uniform distribution and 1 for signal concentrated in a few voxels; CV is the
ratio of SD to mean. Both are **scale-invariant**, so a uniform change in
brightness across a frame leaves them unchanged — which makes them a useful
independent check on the threshold-based M2.

In [ ]:
import re

def get_cell_id(filename):
    """
    Strip the timepoint tag (T0, T1_, T2, ...) from a filename so that all
    frames belonging to the SAME tracked cell/timelapse map to the same id.
    Adjust this regex if it doesn't match your naming convention -- check
    with the verification cell below before trusting downstream results.
    """
    return re.sub(r'_T\d+_?', '_', filename)


def get_fixed_thresholds(all_cells, cell_id_func, baseline_tag='T0',
                          channel='Intensity 3', method=threshold_li):
    """
    Compute one threshold per cell using only that cell's baseline (T0) frame.
    This threshold is then reused for every timepoint of that same cell,
    instead of recomputing an adaptive threshold on every frame.
    """
    thresholds = {}
    for filename, df in all_cells.items():
        if baseline_tag in filename:
            cell_id = cell_id_func(filename)
            thresholds[cell_id] = method(np.array(df[channel]))
    return thresholds


def getMandersColCoeff_fixed(mito_array, green_array, green_threshold):
    """
    M2: fraction of mitochondrial (red) signal sitting in green-positive
    voxels, using a threshold fixed once per cell rather than recomputed
    per frame. Mito channel is used raw (no re-thresholding), since the
    sampled points are already inside the mitochondrial network.
    """
    mito_array = np.asarray(mito_array, dtype=float)
    green_array = np.asarray(green_array, dtype=float)
    denominator = mito_array.sum()
    if denominator == 0:
        return np.nan
    green_mask = green_array > green_threshold
    numerator = mito_array[green_mask].sum()
    return numerator / denominator


def gini(x):
    """Gini coefficient of a 1D intensity array (0 = perfectly uniform,
    approaching 1 = concentrated in very few voxels). Threshold-free."""
    x = np.asarray(x, dtype=float)
    x = x[x >= 0]
    n = len(x)
    if n == 0:
        return np.nan
    x = np.sort(x)
    cum = np.cumsum(x)
    if cum[-1] == 0:
        return np.nan
    return (2 * np.sum(np.arange(1, n + 1) * x) - (n + 1) * cum[-1]) / (n * cum[-1])


def coefficient_of_variation(x):
    """CV = SD / mean of a 1D intensity array. Threshold-free."""
    x = np.asarray(x, dtype=float)
    mean = x.mean()
    if mean == 0:
        return np.nan
    return x.std() / mean


### Verify cell grouping before trusting the fixed threshold

Print out the derived `cell_id` for each filename and eyeball that frames from the same timelapse (different T0/T1/T2...) collapse to the same id, and different cells/replicates stay distinct.

In [ ]:
for filename in sorted(all_cells.keys()):
    print(get_cell_id(filename), '  <-  ', filename)


## 7. Compute per-frame timelapse measurements

Builds one row per image frame, combining threshold-based and threshold-free
measures of signal distribution along the mitochondrial network.

Baseline thresholds are computed first, one per tracked cell from its T0 frame,
then applied to every frame of that cell.

| Measure | Type | Interpretation |
|---|---|---|
| `_manders_fixed` | Threshold-based (M2) | Fraction of mitochondrial signal in second-channel-positive voxels, scored against the cell's T0 threshold |
| `_pearson` | Threshold-free | Linear correlation of the two channels across skeleton nodes |
| `_cv_green` | Threshold-free | SD/mean of the second channel — overall variability |
| `_gini_green` | Threshold-free | Concentration of the second channel signal (0 = uniform, → 1 = focal) |

**Fallback.** If a cell has no T0 frame, `baseline_thresholds.get` falls back to
a per-frame Li threshold for that frame. This keeps the loop running, but such
frames are scored against a different criterion from the rest of the dataset.

Reporting the threshold-based and threshold-free measures side by side is
deliberate: `_gini_green` and `_cv_green` are scale-invariant, so they are
unaffected by uniform intensity changes across the timelapse, and agreement
between them and `_manders_fixed` is evidence that a change in colocalisation is
not an artefact of signal decay.

In [ ]:
baseline_thresholds = get_fixed_thresholds(all_cells, get_cell_id, baseline_tag='T0',
                                            channel='Intensity 3', method=threshold_li)

rows = []
for key in all_cells.keys():
    temp_df = all_cells[key]
    mito = np.array(temp_df['Intensity 2'])
    green = np.array(temp_df['Intensity 3'])

    cell_id = get_cell_id(key)
    # fallback (per-frame Li threshold) only if this cell has no T0 baseline frame
    green_thresh = baseline_thresholds.get(cell_id, threshold_li(green))

    rows.append({
        'Cell': key,
        '_manders_fixed': getMandersColCoeff_fixed(mito, green, green_thresh),
        '_pearson': temp_df['Intensity 2'].corr(temp_df['Intensity 3']),
        '_cv_green': coefficient_of_variation(green),
        '_gini_green': gini(green),
    })

results_fixed = pd.DataFrame(rows)
results_fixed.index = results_fixed['Cell']
results_fixed = results_fixed.iloc[:, 1:]
results_fixed


In [ ]:
df2.to_pickle("")

In [ ]:
df2= pd.read_pickle("H:/DataFrames/PDH/Pda1-NG_CAP_timelapse.pkl")

In [ ]:
def repetition(x):
    if '20260715' in x:
        return 'rep 3'
    elif '20260714' in x:
        return 'rep 2'
    elif '20260708' in x:
        return 'rep 1'
    
def TP(x):
    if "T0" in x:
        return 0
    elif "T1_" in x:
        return 0.5
    elif "T2" in x:
        return 1
    elif "T3" in x:
        return 1.5
    elif "T4" in x:
        return 2
    elif "T5" in x:
        return 2.5
    elif "T6" in x:
        return 3
    elif "T7" in x:
        return 3.5
    elif "T8" in x:
        return 4
    elif "T9" in x:
        return 4.5
    elif "T10" in x:
        return 5
    elif "T11" in x:
        return 5.5
    elif "T12" in x:
        return 6
    elif "T13" in x:
        return 6.5
    elif "T14" in x:
        return 7
    elif "T15" in x:
        return 7.5
    elif "T16" in x:
        return 8
    elif "T17" in x:
        return 8.5
    else:
        return "no time"

In [ ]:
df2=comparison.copy()

In [ ]:
df2["filname"]=df2.index.to_list()

In [ ]:
df2['Replicate']= df2['filname'].apply(repetition)
df2['TP']= df2['filname'].apply(TP)

In [ ]:
df2

In [ ]:
plt.figure()
plt.rcParams['svg.fonttype']='none'
fig, ax = plt.subplots(figsize=(12,4))
sns.boxplot(data=df2,x="TP", y="_manders_fixed", color="gray")

In [ ]:
plt.figure()
plt.rcParams['svg.fonttype']='none'
fig, ax = plt.subplots(figsize=(12,4))
sns.barplot(data=df2,x="TP", y="_manders_fixed", color="gray")

In [ ]:
plt.figure()
plt.rcParams['svg.fonttype']='none'
fig, ax = plt.subplots(figsize=(12,4))
sns.swarmplot(data=df2,x="TP", y="_manders_fixed", color="gray")

In [ ]:
plt.figure()
plt.rcParams['svg.fonttype']='none'
fig, ax = plt.subplots(figsize=(12,4))
sns.boxplot(data=df2,x="TP", y="_manders_fixed", color="gray")

## 8. Manders coefficient over time — chloramphenicol timecourse

SuperPlot of colocalisation across the timelapse, with each timepoint compared
against baseline.

**Layers:** individual frames (small, semi-transparent, coloured by replicate),
replicate means (large, black-outlined), the mean of replicate means (box), and
SD across replicates (error bars).

**Statistics.** Each timepoint is compared against the first timepoint (`ref`)
using a **paired** *t*-test on replicate means, n = 3. Pairing is appropriate
here because the same three biological replicates contribute to every timepoint,
so replicate-to-replicate offsets are removed rather than counted as noise. The
`paired` and `correct` flags at the top make the choice explicit and switchable:
`paired = False` gives Welch's test on unpaired means, and `correct = True`
applies Holm correction across the 15 comparisons.

Degenerate comparisons — fewer than two paired observations, or identical
values — return `NaN` rather than a spurious p-value.

A summary table is printed with the p-value, significance stars, mean difference
from baseline, and the number of paired replicates behind each comparison.
Significance is annotated above each timepoint on the plot.

In [ ]:
df2 = results_fixed 
df2 = df2[df2["TP"] != 8.0]

In [ ]:
nupur2 = ["#BD5B28","#FFD30A","#4d6171","#162734","#b6c9c1","#88A27D"]

In [ ]:


# ---------------------------------------------------------------- data prep
b = df2

c = b.groupby(['TP', 'Replicate'], as_index=False).mean(numeric_only=True)   # replicate means
d = c.groupby('TP', as_index=False).mean(numeric_only=True)                  # mean of replicates
e = c.groupby('TP', as_index=False).std(numeric_only=True)                   # sd of replicates
f = c.pivot_table(columns='TP', values='_manders_fixed', index='Replicate')        # 3 x 16, matched

order = sorted(b['TP'].unique())          # floats, matches f.columns
ref   = order[0]
d = d.set_index('TP').reindex(order)      # guarantee same order as the x-axis
e = e.set_index('TP').reindex(order)

# ---------------------------------------------------------------- statistics
paired  = True      # False -> Welch's t-test
correct = False     # True  -> Holm correction across the 15 comparisons

def stars(p):
    if not np.isfinite(p): return 'n.a.'
    if p < 0.001:          return '***'
    if p < 0.01:           return '**'
    if p < 0.05:           return '*'
    return 'ns'

pvals = []
for tp in order[1:]:
    pair = f[[ref, tp]].dropna()
    a, bb = pair[ref].values, pair[tp].values
    if len(pair) < 2 or np.allclose(a, bb):
        pvals.append(np.nan)
    elif paired:
        pvals.append(stats.ttest_rel(a, bb).pvalue)
    else:
        pvals.append(stats.ttest_ind(f[ref].dropna(), f[tp].dropna(),
                                     equal_var=False).pvalue)

pvals = np.array(pvals, dtype=float)
if correct:
    ok = np.isfinite(pvals)
    pvals[ok] = multipletests(pvals[ok], method='holm')[1]

res = pd.DataFrame({
    'TP': order[1:],
    'p': pvals,
    'sig': [stars(p) for p in pvals],
    'mean_diff': [f[tp].mean() - f[ref].mean() for tp in order[1:]],
    'n': [f[[ref, tp]].dropna().shape[0] for tp in order[1:]],
})

test_name = 'paired t-test' if paired else "Welch's t-test"
print(f"{test_name} vs TP {ref}{' (Holm-corrected)' if correct else ''}\n")
print(res.to_string(index=False,
                    formatters={'p': '{:.4g}'.format,
                                'mean_diff': '{:+.4f}'.format}))

# ---------------------------------------------------------------- plot
plt.rcParams['svg.fonttype'] = 'none'
fig, ax = plt.subplots(figsize=(12, 5))

hue_order = ['rep 1', 'rep 2', 'rep 3']

sns.swarmplot(x='TP', y='_manders_fixed', hue='Replicate', data=b,
              order=order, hue_order=hue_order,
              alpha=0.3, size=3, palette=nupur2, ax=ax)

sns.swarmplot(x='TP', y='_manders_fixed', hue='Replicate', data=c,
              order=order, hue_order=hue_order,
              alpha=0.8, size=10, edgecolor='k', linewidth=1,
              palette=nupur2, ax=ax)

sns.boxplot(x='TP', y='_manders_fixed', data=d, order=order,
            width=0.4, showfliers=False, ax=ax)

plt.errorbar(range(len(order)), d['_manders_fixed'], yerr=e['_manders_fixed'],
             fmt='none', capsize=5, ecolor='k')

ax.get_legend().remove()
ax.set_ylim(0, 1.2)
ax.set(xlabel='Time (h)', ylabel='Manders coefficient')

# significance annotations, no bars
y = ax.get_ylim()[1] * 0.93
for i, p in enumerate(pvals, start=1):
    ax.text(i, y, stars(p), ha='center', va='bottom', fontsize=11)
ax.text(0, y, 'ref', ha='center', va='bottom', fontsize=9, color='grey')

plt.savefig(f"{OUT_DIR}/Pda1-NG_CAP_timecourse.svg", bbox_inches="tight", dpi=300)
plt.show()